## Tarea 4: ¿Cómo vamos a entrenar la RN parte 2?



Instrucciones: 
1. Ponle de nombre a tu tarea `tarea_.ipynb` un Jupyter Notebook (5 points) 
2. Escribe tu nombre comleto y tus iniciales en tu notebook después de la siguiente nota: 
*Este trabajo es mio y sigue la integridad académica del Tec. de Monterrey*
3. Agrega los nombres de todos los compañeros con los que trabajaste. Recuerda que puedes discutir con tus compañeros pero cada quien tiene que entregar su tarea.
4. **ADVERTENCIA!** no se permite el uso de LLMs para estas tareas! El aprendizaje profundo no va a hacer la tarea de aprendizaje profundo... Si no entienden bien los conceptos no hay diferencia entre ustedes y el LLM y esto es sad, se mueren las hadas cuando hacen esto :(
5. Asegurate que tu código corra!
6. Las preguntas de teoria contestalas en texto usando markdown, no comentarios en python




1. Considera una red neuronal donde utilizamos una función de activación sigmoide. Calcula la parcial $\frac{\partial h}{\partial f}$ para esta función de activación. ¿Qué le pasa a esta parcial si el input es un número positivo muy grande? ¿Qué le pasa a esta parcial si es un número negativo muy grande? ¿Qué problema nos genera esto para encontrar los parámetros?


2. Considera una red neuronal con función de activación Heavside para algunas capas intermedias y con función de activación rectangular para otras. Funcionan bien estas funciones de activación para entrenar usando alguna variante de GD? 

$$ Heavside(z) = \begin{cases} 
       0 & z < 0 \\
      1 & z \geq 0 \\
   \end{cases}$$

$$ rect(z) = \begin{cases} 
       0 & z < 0 \\
      1 &  0 \leq z \geq 1 \\
      0 & z > 1
   \end{cases}$$


3. ¿Qué pasaria si inicializamos todos los pesos y los biases de una red en 0?

4. Si tenemos un problema de clasificación múltiple y utilizamos la función multiclase de cross-entropy, la pérdida de entrenamiento alcanzara el 0. Explica tu razonamiento. 

5. Considera el caso cuando la capacidad del modelo es más grande que la cantidad de ejemplos de entrenamiento y que el modelo es lo suficientemente flexible para reducir el error de entrenamiento a cero. ¿Qué implicaciones tiene esto si queremos ajustar un modelo heterocedástico? Propon una manera de solucionar cualquie problema que identifiques. 

6. Usando Pytorch: implementa una red neuronal superficial con una sola capa intermedia y compara que pasa si usas una función de activación sigmoide vs una ReLU. Muestra lo siguiente e interpreta los resultado: 
    
    a. curvas de pérdida del conjunto de entrenamiento y validación 

    b. normas de gradiente por época 

    c. histogramas de los pesos 

    d. cambia la tasa de aprendizaje para encontrar el rango estable 

Acá abajo puedes encontrar una función que genera datos artificiales para la red y una serie de hiperparámetros que tienes que usar


7. Implementa una red neuronal superficial con una capa oculta y realiza los experimentos que se piden acá abajo. Además, implementa y compara la versión con BatchNorm en la capa oculta. Aseurate de ir a la documentación de Pytorch y entender y explicar que hace BatchNorm
    
    a. Implementa una red con lo siguiente `in_dim, hidden, activation, use_batchnorm=False` que si `use_batchnorm=True` inserta `nn.BatchNorm1d(hidden)` justo después de la capa lineal y antes de la activación.
    
    b. Entrena 4 configuraciones distintas: {ReLU, Sigmoid} × {BatchNorm off, BatchNorm on}, usando la misma semilla y split. Guarda train/val loss, grad-norm, param snapshots.

Para cada configuración reporta: 
- curvas train/val
- grad-norm per epoch
- histograms de parámetros (epoch 0, mitad, final)
- resumen final: val-loss y RMSE (si problema de regresión)

    c. Haz un LR-sweep (intenta varias tasas de aprendizaje) por cada configuración y determina la zona estable de lr. Grafica val-loss final vs lr.

    d. Interpretación: compara el efecto de BatchNorm en la dinámica de entrenamiento para cada activación y conecta tus observaciones con los argumentos que vimos en clase sobre propagación de la señal. 

    e. ¿Qué esta haciendo BatchNorm? ¿Por qué nos esta ayudando esto?

Acá abajo vas a encontrar una función que genera datos y los hiperparámetros para esta pregunta



In [ ]:
## Creacion de datos sinteticos para ejercicio 6: 
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_nonlinear_synthetic(seed: int = 42,
                             n_samples: int = 2000,
                             input_dim: int = 1,
                             val_frac: float = 0.2,
                             batch_size: int = 64,
                             heteroscedastic: bool = True):
    """
    Genera datos sintéticos no-lineales para regresión:
      x ~ Uniform(-3, 3) (entrada de dimensión input_dim)
      y = sin(3*x_0) + 0.5 * x_0 + 0.3 * sin(2*x_1)  (si input_dim>1)
      ruido ~ N(0, sigma^2) con sigma creciente con |x_0| si heteroscedastic=True

    Devuelve: train_loader, val_loader, (X_train_np, y_train_np, X_val_np, y_val_np)
    - Los loaders devuelven tensores float32
    - Semilla controlada para reproducibilidad
    """
    rng = np.random.RandomState(seed)

    # Entradas: shape (n_samples, input_dim)
    X = rng.uniform(-3.0, 3.0, size=(n_samples, input_dim)).astype(np.float32)

    # Construcción de una función objetivo no lineal que depende al menos de x[:,0]
    x0 = X[:, 0]
    y = np.sin(3.0 * x0) + 0.5 * x0
    if input_dim > 1:
        # si hay más dimensiones, agregar una dependencia suave en x1
        x1 = X[:, 1]
        y += 0.3 * np.sin(2.0 * x1)

    # Ruido heterocedástico: sigma aumenta con |x0|
    if heteroscedastic:
        sigma = 0.08 + 0.25 * (np.abs(x0) / np.max(np.abs(x0)))
    else:
        sigma = np.full_like(x0, 0.2)

    noise = rng.normal(0.0, sigma).astype(np.float32)
    y = (y + noise).astype(np.float32).reshape(-1, 1)

    # split train / val
    n_val = int(n_samples * val_frac)
    idx = np.arange(n_samples)
    rng.shuffle(idx)
    tr_idx = idx[n_val:]
    va_idx = idx[:n_val]

    X_train = X[tr_idx]
    y_train = y[tr_idx]
    X_val = X[va_idx]
    y_val = y[va_idx]

    # DataLoaders (torch tensors)
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # También devolvemos numpy arrays para evaluaciones fuera de torch si se desea
    return train_loader, val_loader, (X_train, y_train, X_val, y_val)

SEED = 1234                 # semilla fija para que todos obtengan resultados similares
N_SAMPLES = 2000
INPUT_DIM = 1               # puedes poner 2 si quieres un problema 2D
BATCH_SIZE = 64
EPOCHS = 80                 # suficiente para ver convergencia en este toy problem
HIDDEN = 64                 # neuronas en la capa oculta
LR = 1e-2                   # tasa de aprendizaje inicial para comparar ReLU vs Sigmoid



def find_device():
    if torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    elif torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    else:
        DEVICE = torch.device("cpu")
    return DEVICE

DEVICE = find_device()


In [ ]:
## para la pregunta 7
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_nonlinear_synthetic_v2(seed: int = 2025,
                                n_samples: int = 2500,
                                input_dim: int = 1,
                                val_frac: float = 0.2,
                                batch_size: int = 64,
                                heteroscedastic: bool = True):
    """
    Generador reproducible para la pregunta 7.
    - no linealidad: mezcla de sinusoides y término cuadrático para forzar no-linealidad.
    - heteroscedastic: ruido con varianza que aumenta con |x|.
    Devuelve: train_loader, val_loader, (X_train, y_train, X_val, y_val) (numpy arrays para checks)
    """
    rng = np.random.RandomState(seed)

    X = rng.uniform(-3.0, 3.0, size=(n_samples, input_dim)).astype(np.float32)
    x0 = X[:, 0]

    # función objetivo no-lineal (mezcla de sin y cuadrático)
    y = 0.8 * np.sin(2.5 * x0) + 0.4 * x0 - 0.2 * (x0**2)

    if input_dim > 1:
        # si hay más dimensiones añadimos interacción ligera
        x1 = X[:, 1]
        y += 0.3 * np.sin(1.5 * x1) + 0.1 * x0 * x1

    # ruido heterocedástico: sigma crece con |x0|
    if heteroscedastic:
        sigma = 0.05 + 0.25 * (np.abs(x0) / np.max(np.abs(x0)))
    else:
        sigma = np.full_like(x0, 0.15)

    noise = rng.normal(0.0, sigma).astype(np.float32)
    y = (y + noise).astype(np.float32).reshape(-1, 1)

    # split
    n_val = int(n_samples * val_frac)
    idx = np.arange(n_samples)
    rng.shuffle(idx)
    tr_idx = idx[n_val:]
    va_idx = idx[:n_val]

    X_train = X[tr_idx]; y_train = y[tr_idx]
    X_val   = X[va_idx]; y_val   = y[va_idx]

    # DataLoaders
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, (X_train, y_train, X_val, y_val)



 # Parámetros reproducibles para los estudiantes
SEED = 2025
N_SAMPLES = 2500
INPUT_DIM = 1
BATCH_SIZE = 64
EPOCHS = 80
HIDDEN = 64
LR_DEFAULT = 1e-2
   
